# Caustic structure in Kerr spacetime — sky-plane critical curves

This notebook visualises the output of `caustic_discplane`, which traces rays backward
from the observer's image plane to the accretion disc and computes the Jacobian
$J = \partial(r_{\rm disc},\,\phi_{\rm disc}) / \partial(x_{\rm sky},\,y_{\rm sky})$
at each pixel via central finite differences.

**Critical curves** on the sky are the zero-contours of $\det J$.  Each critical curve
separates image orders (direct image $n=0$, first photon ring $n=1$, ...).
The corresponding locus on the disc is the **caustic**.

FITS extensions used here:
- `ORDER` — number of radial turning points (`rdot_flips`); labels image order
- `DET_J` — Jacobian determinant; zero-crossings = critical curves
- `SIGN_J` — sign of $\det J$; parity flips at each critical curve
- `HIT` — disc-hit mask
- `RADIUS`, `PHI` — disc coordinates of each image-plane pixel
- `REDSHIFT` — photon energy ratio $E_{\rm obs}/E_{\rm emit}$

In [ ]:
import numpy as np
import astropy.io.fits as pyfits
import matplotlib.pyplot as plt
import matplotlib.colors as colors
import matplotlib.ticker as ticker
from matplotlib.colors import BoundaryNorm, ListedColormap

%matplotlib inline

In [ ]:
fits_filename = '../dat/caustic_discplane.fits'

with pyfits.open(fits_filename) as f:
    hdr    = f[1].header          # first extension header carries axes
    det_J  = np.array(f['DET_J'].data)
    sign_J = np.array(f['SIGN_J'].data)
    order  = np.array(f['ORDER'].data)
    hit    = np.array(f['HIT'].data)
    radius = np.array(f['RADIUS'].data)
    phi    = np.array(f['PHI'].data)
    redshift = np.array(f['REDSHIFT'].data)

    x0   = f['DET_J'].header['X0']
    xmax = f['DET_J'].header['XMAX']
    y0   = f['DET_J'].header['Y0']
    ymax = f['DET_J'].header['YMAX']
    spin = f[0].header['SPIN']
    incl = f[0].header['INCL']
    isco = f[0].header['ISCO']

# Image extent for imshow
extent = [x0, xmax, y0, ymax]

print(f'Spin a/M = {spin},  inclination = {incl}°,  ISCO = {isco:.3f} rg')
print(f'Image plane: {det_J.shape[0]} × {det_J.shape[1]} pixels,  extent {x0}…{xmax} × {y0}…{ymax} rg')
print(f'Image orders present: {np.unique(order[hit == 1])}')
print(f'Disc-hit pixels: {int(hit.sum())} / {hit.size}')

## 1. Image order map

The `ORDER` (= `rdot_flips`) map shows which image of the disc each sky pixel belongs to.
Each distinct colour band is separated from the next by a critical curve.
The innermost black region is the photon capture cross-section (the shadow).

In [ ]:
# Colour each image order distinctly; grey = no disc hit
order_display = order.copy().astype(float)
order_display[hit == 0] = np.nan

max_order = int(np.nanmax(order_display))
cmap_order = plt.cm.get_cmap('tab10', max_order + 1)
norm_order = colors.BoundaryNorm(np.arange(-0.5, max_order + 1.5), max_order + 1)

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(order_display, origin='lower', extent=extent,
               cmap=cmap_order, norm=norm_order, interpolation='nearest')
cbar = fig.colorbar(im, ax=ax, ticks=np.arange(0, max_order + 1))
cbar.set_label('Image order $n$ (rdot_flips)', fontsize=11)
ax.set_xlabel('Image plane $x$ (r$_g$)', fontsize=12)
ax.set_ylabel('Image plane $y$ (r$_g$)', fontsize=12)
ax.set_title(f'Image order map  |  spin = {spin}, incl = {incl}°', fontsize=13)
plt.tight_layout()
plt.savefig('../dat/caustic_order_map.pdf', bbox_inches='tight')
plt.show()

## 2. Jacobian determinant $\det J$

$\det J$ measures the area ratio between a patch on the image plane and the
corresponding patch on the disc.  It diverges at the photon ring edges (infinite
magnification) and changes sign at each critical curve (image parity flip).

- **Zero-crossings** within a single image order are fold-type critical curves.
- **Boundary pixels** between different image orders are also critical curves
  (marked with a large sentinel value $10^{30}$ and shown separately).

In [ ]:
SENTINEL = 1e30

# Mask: only pixels with a disc hit, finite det_J, and not the sentinel
valid = (hit == 1) & np.isfinite(det_J) & (np.abs(det_J) < SENTINEL * 0.5)

det_J_plot = np.full_like(det_J, np.nan)
det_J_plot[valid] = det_J[valid]

# Clip at a symmetric percentile to tame divergences near the rings
vmax = np.nanpercentile(np.abs(det_J_plot[valid]), 95)
print(f'Colour scale ±{vmax:.3f} (95th percentile of |det J|)')

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(det_J_plot, origin='lower', extent=extent,
               cmap='RdBu_r', norm=colors.Normalize(vmin=-vmax, vmax=vmax),
               interpolation='nearest')
cbar = fig.colorbar(im, ax=ax)
cbar.set_label(r'$\det J$', fontsize=12)
ax.set_xlabel('Image plane $x$ (r$_g$)', fontsize=12)
ax.set_ylabel('Image plane $y$ (r$_g$)', fontsize=12)
ax.set_title(r'Jacobian determinant $\det J$  |  critical curves at $\det J = 0$', fontsize=12)
plt.tight_layout()
plt.savefig('../dat/caustic_detJ.pdf', bbox_inches='tight')
plt.show()

## 3. Critical curves on the sky

Critical curves are extracted as the zero-contours of $\det J$ (within each image order)
plus the boundaries between image orders (where `ORDER` changes between adjacent pixels).
Both are overlaid on the image order map.

In [ ]:

# Pixel coordinate axes
Ny, Nx = det_J.shape
xvals = np.linspace(x0, xmax, Nx)
yvals = np.linspace(y0, ymax, Ny)

# Construct an "order-boundary" mask: pixels adjacent to a different-order pixel
order_boundary = np.zeros_like(order, dtype=bool)
order_hit = order.copy()
order_hit[hit == 0] = -99   # sentinel for no-hit
order_boundary[1:,  :] |= (order_hit[1:,  :] != order_hit[:-1, :])
order_boundary[:-1, :] |= (order_hit[:-1, :] != order_hit[1:,  :])
order_boundary[:,  1:] |= (order_hit[:,  1:] != order_hit[:, :-1])
order_boundary[:, :-1] |= (order_hit[:, :-1] != order_hit[:,  1:])
# Only count boundaries within disc-hit region
order_boundary &= (hit == 1)

fig, ax = plt.subplots(figsize=(8, 8))

# Background: image order map
im = ax.imshow(order_display, origin='lower', extent=extent,
               cmap=cmap_order, norm=norm_order,
               interpolation='nearest', alpha=0.7)

# Critical curves from zero-crossings of det_J within same image order
ax.contour(xvals, yvals, det_J_plot, levels=[0],
           colors='white', linewidths=1.2, linestyles='-')

# Critical curves from image-order boundaries
ob_display = np.where(order_boundary, 1.0, np.nan)
ax.imshow(ob_display, origin='lower', extent=extent,
          cmap='autumn', vmin=0, vmax=1, alpha=0.9,
          interpolation='nearest')

cbar = fig.colorbar(im, ax=ax, ticks=np.arange(0, max_order + 1), fraction=0.046, pad=0.04)
cbar.set_label('Image order $n$', fontsize=11)

from matplotlib.lines import Line2D
from matplotlib.patches import Patch
legend_elements = [
    Line2D([0], [0], color='white', lw=1.5, label=r'Critical curve ($\det J = 0$, within order)'),
    Patch(facecolor='red', alpha=0.8, label='Image-order boundary (also a critical curve)'),
]
ax.legend(handles=legend_elements, loc='upper right', fontsize=9,
          framealpha=0.7, facecolor='0.15', labelcolor='white')

ax.set_xlabel('Image plane $x$ (r$_g$)', fontsize=12)
ax.set_ylabel('Image plane $y$ (r$_g$)', fontsize=12)
ax.set_title(f'Critical curves on the sky  |  spin = {spin}, incl = {incl}°', fontsize=13)
plt.tight_layout()
plt.savefig('../dat/caustic_critical_curves.pdf', bbox_inches='tight')
plt.show()


## 4. Zoom into the photon ring region

The first photon ring ($n=1$) is a very narrow band just outside the shadow.
Zooming in reveals the banding and the tight critical curves bounding it.
Adjust `zoom_half` to change the zoom level.

In [ ]:
# Rough size of the photon shadow for spin~1 is ~5 rg impact parameter.
# Zoom to ±8 rg to show the shadow + first ring.
zoom_half = 8.0

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, data, title, kw in zip(
    axes,
    [order_display, det_J_plot],
    ['Image order map (zoom)', r'$\det J$ (zoom)'],
    [dict(cmap=cmap_order, norm=norm_order),
     dict(cmap='RdBu_r', norm=colors.Normalize(vmin=-vmax, vmax=vmax))]):

    im = ax.imshow(data, origin='lower', extent=extent, interpolation='nearest', **kw)
    ax.contour(xvals, yvals, det_J_plot, levels=[0],
               colors='white', linewidths=0.8, linestyles='-')
    ax.set_xlim(-zoom_half, zoom_half)
    ax.set_ylim(-zoom_half, zoom_half)
    ax.set_xlabel('Image plane $x$ (r$_g$)', fontsize=11)
    ax.set_ylabel('Image plane $y$ (r$_g$)', fontsize=11)
    ax.set_title(title, fontsize=12)
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig.suptitle(f'Photon ring region  |  spin = {spin}, incl = {incl}°  (zoom ±{zoom_half} r$_g$)', fontsize=13)
plt.tight_layout()
plt.savefig('../dat/caustic_photon_ring_zoom.pdf', bbox_inches='tight')
plt.show()

## 5. Disc coordinate and redshift maps

For context, the disc radius, azimuthal angle, and observed redshift at each
image-plane pixel.  These will be used in Phase 2 to map caustic positions to
emission-line energies and reverberation time delays.

In [ ]:
radius_plot   = np.where(hit == 1, radius,   np.nan)
phi_plot      = np.where(hit == 1, phi,      np.nan)
redshift_plot = np.where(hit == 1, redshift, np.nan)

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

im0 = axes[0].imshow(radius_plot, origin='lower', extent=extent,
                     norm=colors.LogNorm(vmin=isco, vmax=50), cmap='viridis',
                     interpolation='nearest')
axes[0].set_title('Disc radius (r$_g$)', fontsize=12)
fig.colorbar(im0, ax=axes[0], label='r (r$_g$)')

im1 = axes[1].imshow(phi_plot, origin='lower', extent=extent,
                     norm=colors.Normalize(vmin=-np.pi, vmax=np.pi), cmap='twilight',
                     interpolation='nearest')
axes[1].set_title('Disc azimuthal angle $\\phi$ (rad)', fontsize=12)
fig.colorbar(im1, ax=axes[1], label='$\\phi$ (rad)')

im2 = axes[2].imshow(redshift_plot, origin='lower', extent=extent,
                     norm=colors.Normalize(vmin=0.2, vmax=1.5), cmap='RdYlBu_r',
                     interpolation='nearest')
axes[2].set_title('Redshift $E_{\\rm obs}/E_{\\rm emit}$', fontsize=12)
fig.colorbar(im2, ax=axes[2], label='$E_{\\rm obs}/E_{\\rm emit}$')

for ax in axes:
    # Overlay critical curve contour
    ax.contour(xvals, yvals, det_J_plot, levels=[0],
               colors='white', linewidths=0.7, linestyles='--', alpha=0.6)
    ax.set_xlabel('Image plane $x$ (r$_g$)', fontsize=11)
    ax.set_ylabel('Image plane $y$ (r$_g$)', fontsize=11)

fig.suptitle(f'Disc coordinate and redshift maps  |  spin = {spin}, incl = {incl}°', fontsize=13)
plt.tight_layout()
plt.savefig('../dat/caustic_disc_maps.pdf', bbox_inches='tight')
plt.show()

## 6. Parity map — sign of $\det J$

The sign of the Jacobian determinant tracks image parity.  Each time a critical
curve is crossed the parity flips ($+1 \to -1$ or vice versa).  The direct image
($n=0$) has positive parity; the first photon ring ($n=1$) has negative parity;
and so on alternately.

In [ ]:
sign_plot = sign_J.copy().astype(float)
sign_plot[hit == 0] = np.nan
sign_plot[np.abs(det_J) >= SENTINEL * 0.5] = np.nan  # hide sentinels
sign_plot[~np.isfinite(det_J)] = np.nan

fig, ax = plt.subplots(figsize=(7, 7))
im = ax.imshow(sign_plot, origin='lower', extent=extent,
               cmap='coolwarm', norm=colors.Normalize(vmin=-1.2, vmax=1.2),
               interpolation='nearest')
ax.contour(xvals, yvals, det_J_plot, levels=[0],
           colors='black', linewidths=1.0, linestyles='-')
cbar = fig.colorbar(im, ax=ax, ticks=[-1, 0, 1])
cbar.set_label('sign($\\det J$)  [image parity]', fontsize=11)
ax.set_xlabel('Image plane $x$ (r$_g$)', fontsize=12)
ax.set_ylabel('Image plane $y$ (r$_g$)', fontsize=12)
ax.set_title('Image parity  |  critical curves in black', fontsize=12)
plt.tight_layout()
plt.savefig('../dat/caustic_parity.pdf', bbox_inches='tight')
plt.show()

## 7. Critical curve statistics

Locate pixels that lie on or adjacent to a critical curve and summarise the
disc coordinates they map to.

In [ ]:
# Pixels on a critical curve: sign of det_J changes between this pixel and a neighbour
# (within the same image order and both having finite det_J)
is_cc = np.zeros_like(hit, dtype=bool)

for shift_axis, shift_dir in [((0,1), (slice(1,None), slice(None))),
                               ((0,1), (slice(None,-1), slice(None))),
                               ((1,0), (slice(None), slice(1,None))),
                               ((1,0), (slice(None), slice(None,-1)))]:
    pass  # will compute below

sJ = sign_J.copy()
sJ[~np.isfinite(det_J)] = 0
sJ[np.abs(det_J) >= SENTINEL * 0.5] = 0
sJ[hit == 0] = 0

# A pixel is on a critical curve if any neighbour has opposite sign
is_cc[1:, :]  |= (sJ[1:,  :] * sJ[:-1, :] < 0) & (order[1:, :] == order[:-1, :])
is_cc[:-1, :] |= (sJ[:-1, :] * sJ[1:,  :] < 0) & (order[:-1, :] == order[1:, :])
is_cc[:, 1:]  |= (sJ[:, 1:] * sJ[:, :-1] < 0)  & (order[:, 1:] == order[:, :-1])
is_cc[:, :-1] |= (sJ[:, :-1] * sJ[:, 1:]  < 0) & (order[:, :-1] == order[:, 1:])
# Also add the order-boundary pixels
is_cc |= order_boundary

print(f'Pixels identified on critical curves: {is_cc.sum()}')
print()
for n in range(max_order + 1):
    mask_n = is_cc & (order == n)
    if mask_n.sum() == 0:
        continue
    r_cc  = radius[mask_n]
    rs_cc = redshift[mask_n]
    print(f'  n={n}: {mask_n.sum():5d} pixels,  '
          f'r_disc ∈ [{r_cc.min():.2f}, {r_cc.max():.2f}] rg,  '
          f'redshift ∈ [{rs_cc.min():.3f}, {rs_cc.max():.3f}]')